# Laboratory 3 — Part A: Feature Engineering for Text Data
Bag-of-Words with `CountVectorizer`, plus Home-Lab Exercises 1 and 2.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The sun is shining today.",
    "The weather is good, the sun is great.",
    "A sunny day is a wonderful day."
]
print("--- Raw Text Documents ---")
for i, doc in enumerate(documents):
    print(f"Doc {i+1}: {doc}")

--- Raw Text Documents ---
Doc 1: The sun is shining today.
Doc 2: The weather is good, the sun is great.
Doc 3: A sunny day is a wonderful day.


## Task 3.1 — Vectorising text with `CountVectorizer`

In [2]:
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()
text_matrix = X_text.toarray()

print("--- Bag-of-Words (BoW) Transformation ---")
print(f"Vocabulary (Features): {feature_names}")
print(f"Shape of Feature Matrix: {text_matrix.shape}   (documents x unique terms)")
print(text_matrix)

--- Bag-of-Words (BoW) Transformation ---
Vocabulary (Features): ['day' 'good' 'great' 'is' 'shining' 'sun' 'sunny' 'the' 'today' 'weather'
 'wonderful']
Shape of Feature Matrix: (3, 11)   (documents x unique terms)
[[0 0 0 1 1 1 0 1 1 0 0]
 [0 1 1 2 0 1 0 2 0 1 0]
 [2 0 0 1 0 0 1 0 0 0 1]]


In [3]:
df_text = pd.DataFrame(text_matrix, columns=feature_names,
                       index=[f"Doc {i+1}" for i in range(len(documents))])
print("BoW Numerical Feature Matrix (Machine Understandable Format):\n")
display(df_text)

BoW Numerical Feature Matrix (Machine Understandable Format):



,day,good,great,is,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,1,1,1,0,1,1,0,0
Doc 2,0,1,1,2,0,1,0,2,0,1,0
Doc 3,2,0,0,1,0,0,1,0,0,0,1


In [4]:
print("Is 'a' in the vocabulary? ", 'a' in feature_names)
print("Is 'the' in the vocabulary?", 'the' in feature_names)
print()
print("Default token pattern :", vectorizer.token_pattern)
print("Lowercase enabled     :", vectorizer.lowercase)
print()
print("The token pattern requires 2+ word characters, which is why 'a' is dropped.")

Is 'a' in the vocabulary?  False
Is 'the' in the vocabulary? True

Default token pattern : (?u)\b\w\w+\b
Lowercase enabled     : True

The token pattern requires 2+ word characters, which is why 'a' is dropped.


In [5]:
total_cells = text_matrix.size
zero_cells = (text_matrix == 0).sum()

print(f"Matrix cells : {total_cells}")
print(f"Zero cells   : {zero_cells}")
print(f"Sparsity     : {zero_cells / total_cells:.1%}")
print()
print("With a real corpus of 20,000 documents this figure exceeds 99%,")
print("which is why scikit-learn returns a SPARSE matrix, not a dense array.")
print("Type returned by fit_transform:", type(X_text))

Matrix cells : 33
Zero cells   : 18
Sparsity     : 54.5%

With a real corpus of 20,000 documents this figure exceeds 99%,
which is why scikit-learn returns a SPARSE matrix, not a dense array.
Type returned by fit_transform: <class 'scipy.sparse._csr.csr_matrix'>


## Discussion questions (Part A) — Answers

**1. What does each column in the resulting matrix represent?**
Each column represents **one unique term (word) from the vocabulary** learned from the whole corpus — i.e. one *feature*. The value in a cell is the **number of times that term occurs in that document**. Here the vocabulary has 11 terms, so every document becomes a point in an 11-dimensional space. Columns are ordered alphabetically (`day, good, great, is, shining, sun, sunny, the, today, weather, wonderful`).

**2. In this specific numerical matrix, what is the machine-understandable format?**
The machine-understandable format is the **document–term matrix**: a $3 \times 11$ matrix of non-negative integers in which each **row is a fixed-length numeric vector** describing one document (e.g. Doc 1 = `[0,0,0,1,1,1,0,1,1,0,0]`). Documents that originally had different lengths and structure are all mapped onto vectors of the same length (11), so algorithms such as Naive Bayes, Logistic Regression or SVM can consume them. Because most entries are 0 (here 18 of 33 cells ≈ 54.5 %, and >99 % for real corpora), scikit-learn stores it as a **sparse matrix**.

**3. How is this format an oversimplification of the original text?**
Information that is **discarded**:
- **Word order and grammar** — "dog bites man" and "man bites dog" get identical vectors.
- **Context and meaning** — *sun* and *sunny* are unrelated columns; synonyms are not linked; negation ("not good") is invisible.
- **Case and punctuation** — everything is lowercased, punctuation removed (*The*/*the* merge).
- **Short tokens** — the single-character word *"a"* is silently dropped by the default `token_pattern`.
- **Relative importance** — raw counts treat frequent function words (*the*, *is*) as equally informative as rare content words (*shining*); a longer document gets larger numbers simply because it is longer.
- **Unseen words** — any word not in the training vocabulary is ignored at prediction time.

---
# Home-Lab Exercise 1 — `ngram_range=(1,2)`

In [6]:
vec_uni = CountVectorizer()                       # unigrams only
vec_bi  = CountVectorizer(ngram_range=(1, 2))     # unigrams + bigrams

vec_uni.fit(documents)
X_bi = vec_bi.fit_transform(documents)

names_bi = vec_bi.get_feature_names_out()
n_uni = len(vec_uni.get_feature_names_out())
n_all = len(names_bi)
n_bigrams = sum(' ' in n for n in names_bi)

print(f"Unigram-only vocabulary size : {n_uni}")
print(f"ngram_range=(1,2) vocabulary : {n_all}")
print(f"   of which unigrams : {n_all - n_bigrams}")
print(f"   of which bigrams  : {n_bigrams}")
print()
print(names_bi)
display(pd.DataFrame(X_bi.toarray(), columns=names_bi,
                     index=[f"Doc {i+1}" for i in range(len(documents))]))

Unigram-only vocabulary size : 11
ngram_range=(1,2) vocabulary : 24
   of which unigrams : 11
   of which bigrams  : 13

['day' 'day is' 'good' 'good the' 'great' 'is' 'is good' 'is great'
 'is shining' 'is wonderful' 'shining' 'shining today' 'sun' 'sun is'
 'sunny' 'sunny day' 'the' 'the sun' 'the weather' 'today' 'weather'
 'weather is' 'wonderful' 'wonderful day']


,day,day is,good,good the,great,is,is good,is great,is shining,is wonderful,...,sunny,sunny day,the,the sun,the weather,today,weather,weather is,wonderful,wonderful day
Doc 1,0,0,0,0,0,1,0,0,1,0,...,0,0,1,1,0,1,0,0,0,0
Doc 2,0,0,1,1,1,2,1,1,0,0,...,0,0,2,1,1,0,1,1,0,0
Doc 3,2,1,0,0,0,1,0,0,0,1,...,1,1,0,0,0,0,0,0,1,1


**Result and explanation.** The vocabulary grows from **11 to 24 features** (11 unigrams + 13 bigrams).

A bigram is added for **every distinct pair of adjacent tokens** in the corpus. A document with *n* tokens contributes *n − 1* bigrams, so the number of bigram features grows roughly **linearly with corpus size** — and, because most word pairs occur only once, almost every bigram is a brand-new column. On a real corpus with a vocabulary of *V* words, the number of possible bigrams is up to *V²*, so the matrix becomes much wider and much sparser. The benefit is that local word order is partly recovered (*"sunny day"*, *"is good"*); the cost is dimensionality, memory and sparsity.

---
# Home-Lab Exercise 2 — a fourth document repeating *sun* five times

In [7]:
docs4 = documents + ["sun sun sun sun sun"]
vec4 = CountVectorizer()
X4 = vec4.fit_transform(docs4).toarray()
names4 = vec4.get_feature_names_out()
df4 = pd.DataFrame(X4, columns=names4, index=[f"Doc {i+1}" for i in range(len(docs4))])
display(df4)

print("Vocabulary unchanged? ", list(names4) == list(feature_names))
print("Column 'sun' before  :", df_text['sun'].tolist())
print("Column 'sun' after   :", df4['sun'].tolist())
print()
print("Row sums (total tokens kept per document):", X4.sum(axis=1).tolist())

,day,good,great,is,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,1,1,1,0,1,1,0,0
Doc 2,0,1,1,2,0,1,0,2,0,1,0
Doc 3,2,0,0,1,0,0,1,0,0,0,1
Doc 4,0,0,0,0,0,5,0,0,0,0,0


Vocabulary unchanged?  True
Column 'sun' before  : [1, 1, 0]
Column 'sun' after   : [1, 1, 0, 5]

Row sums (total tokens kept per document): [5, 8, 5, 5]


In [8]:
# Why raw counts favour long documents: compare 'sun' evidence in a short vs a long doc
short_doc = "sun sun"
long_doc  = ("sun " * 2) + ("filler word " * 50)   # same 2 mentions of 'sun', but 100 extra words
v = CountVectorizer().fit([short_doc, long_doc])
M = v.transform([short_doc, long_doc]).toarray()
print(dict(zip(v.get_feature_names_out(), M[0])), "<- short doc")
print(dict(zip(v.get_feature_names_out(), M[1])), "<- long doc")
print()

# Two common fixes: L1-normalise the counts (relative frequency) or use TF-IDF
from sklearn.preprocessing import normalize
print("L1-normalised rows (relative frequency):")
print(np.round(normalize(X4.astype(float), norm='l1'), 2))

{'filler': np.int64(0), 'sun': np.int64(2), 'word': np.int64(0)} <- short doc
{'filler': np.int64(50), 'sun': np.int64(2), 'word': np.int64(50)} <- long doc

L1-normalised rows (relative frequency):
[[0.   0.   0.   0.2  0.2  0.2  0.   0.2  0.2  0.   0.  ]
 [0.   0.12 0.12 0.25 0.   0.12 0.   0.25 0.   0.12 0.  ]
 [0.4  0.   0.   0.2  0.   0.   0.2  0.   0.   0.   0.2 ]
 [0.   0.   0.   0.   0.   1.   0.   0.   0.   0.   0.  ]]


**Result and explanation.** Adding `"sun sun sun sun sun"` adds **one new row**; the vocabulary does not change (no new word), and the `sun` column becomes `[1, 1, 0, 5]`. Doc 4 now has the largest value in the whole matrix even though it contains no information except a single repeated word.

**Raw counts favour long documents** because a count is an *absolute* number: the longer a document is, the more often every word is likely to occur, so long documents get larger entries and larger vector norms. A distance- or dot-product-based model (kNN, SVM, cosine before normalisation) then treats "length" as if it were "relevance". The repetition of *sun* five times also makes Doc 4 look five times "more about the sun" than Doc 1, although it just repeats a word. Remedies are **normalising counts** (relative frequency, L1/L2 norm), **TF-IDF** weighting, or **binary** presence (`CountVectorizer(binary=True)`).